# SubGen on Colab

Sentence-level `.srt` subtitles from English speech, using WhisperX.

1. **Runtime → Change runtime type → T4 GPU** (or any GPU).
2. Run the cells in order. The last one prints a public `gradio.live` link; open it to use the app.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
![ -d SubGen ] || git clone https://github.com/enoky/SubGen.git
%cd /content/SubGen
!git pull --ff-only

In [ ]:
%%bash
# Install into Colab's own Python, keeping its preinstalled CUDA build of torch when
# WhisperX accepts it (the repo's lockfile targets Windows CUDA 13.2 wheels).
pip install -q uv
python -c "import torch, torchaudio; print(f'torch=={torch.__version__}'); print(f'torchaudio=={torchaudio.__version__}')" > /tmp/torch-pins.txt
PKGS=("whisperx>=3.7.2" "deepmultilingualpunctuation>=1.0.1" "spacy>=3.8,<3.9" "srt>=3.5.3" "tqdm" "gradio" "pillow"
      "en-core-web-lg @ https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.8.0/en_core_web_lg-3.8.0-py3-none-any.whl")
uv pip install --system -q -c /tmp/torch-pins.txt "${PKGS[@]}" \
  || { echo "Colab's torch is too old for WhisperX; letting it upgrade torch."; uv pip install --system -q "${PKGS[@]}"; }

### Optional: cache models on Google Drive

Run this, then put `/content/drive/MyDrive/subgen-models` in the app's **Model cache directory** setting so the Whisper model is not re-downloaded every session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/SubGen
from subgen_gradio import build_app

build_app().queue().launch(share=True, debug=True)